# 2 · Entity resolution: one supplier, two records

Master data from two sources holds duplicates that rules miss (*Gem. Amsterdam* vs *Gemeente Amsterdam, afd. Inkoop*) and false matches that fuzzy matching creates (two different *Jansen B.V.* companies).

**The pattern:** code normalizes, blocks and settles the obvious pairs → Jev judges the rest → its probability decides *merge*, *keep separate* or *review*.

In [ ]:
import json

import pandas as pd

from jevdemo import er
from jevdemo.judge import MODEL, Judge, LiveUnavailable, NotRecorded, request_body
from jevdemo.pipeline import bucket

judge = Judge()  # JEV_MODE=live for live calls; default replays recordings
print(f"mode: {judge.mode} · model: {MODEL}")

## Two supplier tables
Synthetic. `er_gold.csv` holds our hand labels: which pairs are the same organisation.

In [ ]:
a, b, gold = er.load_table("a"), er.load_table("b"), er.load_gold()
display(pd.DataFrame(a).head(8), pd.DataFrame(b).head(8))

## Code step 1: normalize and block
Only pairs that share a postcode, a KvK prefix or a phonetic name key become candidates. Blocking recall caps everything after it: Jev can't match a pair it never sees.

In [ ]:
pairs = er.candidate_pairs(a, b)
seen = {(ra["id"], rb["id"]) for ra, rb in pairs}
print(
    f"{len(a)} × {len(b)} = {len(a) * len(b)} possible pairs → {len(pairs)} candidates"
)
print(
    f"blocking recall: {len(seen & gold)} of {len(gold)} gold pairs; never paired: {sorted(gold - seen)}"
)

## Code step 2: exact rules settle the obvious
Same KvK number or IBAN → match, no Jev needed.

In [ ]:
exact = [(ra, rb) for ra, rb in pairs if er.exact_match(ra, rb)]
ambiguous = [(ra, rb) for ra, rb in pairs if not er.exact_match(ra, rb)]
print(
    f"exact matches: {[(ra['id'], rb['id']) for ra, rb in exact]} · left for Jev: {len(ambiguous)}"
)

## What Jev receives
Both records as named JSON, and one Noul whose criteria say what counts as *the same organisation*.

In [ ]:
jansen = next(p for p in ambiguous if p[0]["id"] == "A02")
print(json.dumps(request_body(*er.pair_request(*jansen)), indent=2, ensure_ascii=False))

## Jev judges every remaining pair

In [ ]:
answers = judge.ask_many([er.pair_request(ra, rb) for ra, rb in ambiguous])
p_same = {
    (ra["id"], rb["id"]): ans.response.nouls["same"].noul
    for (ra, rb), ans in zip(ambiguous, answers)
}

## The match table
`p_same ≥ HI` → merge, `≤ LO` → keep separate, in between → review. These thresholds were picked on this same small set, with no holdout; on client data you'd set them on labelled pairs first.

In [ ]:
LO, HI = er.LO, er.HI

names = {r["id"]: r["name"] for r in a + b}
matches = [
    {
        "id_a": ra["id"],
        "id_b": rb["id"],
        "p_same": 1.0,
        "decision": "accept",
        "by": "exact rule",
    }
    for ra, rb in exact
]
matches += [
    {"id_a": x, "id_b": y, "p_same": p, "decision": bucket(p, LO, HI), "by": "Jev"}
    for (x, y), p in p_same.items()
]
table = pd.DataFrame(matches)
table.insert(2, "name_a", table["id_a"].map(names))
table.insert(3, "name_b", table["id_b"].map(names))
table["gold"] = [(x, y) in gold for x, y in zip(table["id_a"], table["id_b"])]
table.sort_values("p_same", ascending=False)

## Pairs become clusters
Union-find over the merges gives every record a cluster id.

In [ ]:
merges = [(m["id_a"], m["id_b"]) for m in matches if m["decision"] == "accept"]
cluster_of = er.clusters([r["id"] for r in a + b], merges)
grouped = pd.DataFrame(
    [{"cluster": c, "id": i, "name": names[i]} for i, c in cluster_of.items()]
)
grouped[grouped.duplicated("cluster", keep=False)].sort_values(["cluster", "id"])

## Jev vs the fuzzy-match baseline
Same candidates, same exact rules. The baseline is Levenshtein similarity on normalized names: ≥ 0.85 merge, ≤ 0.50 keep separate.

In [ ]:
jev = {(m["id_a"], m["id_b"]): m["decision"] for m in matches}
fuzzy = {
    (ra["id"], rb["id"]): "accept"
    if er.exact_match(ra, rb)
    else bucket(er.name_similarity(ra, rb), 0.50, 0.85)
    for ra, rb in pairs
}
pd.DataFrame({"Levenshtein": er.evaluate(fuzzy, gold), "Jev": er.evaluate(jev, gold)}).T

## Where they disagree

In [ ]:
diff = [
    {
        "pair": f"{names[x]} | {names[y]}",
        "gold": (x, y) in gold,
        "Levenshtein": fuzzy[(x, y)],
        "Jev": jev[(x, y)],
    }
    for x, y in jev
    if fuzzy[(x, y)] != jev[(x, y)]
]
pd.DataFrame(diff)

## Your turn
Edit the two records. The preset is recorded; a changed pair needs `JEV_MODE=live`.

In [ ]:
record_a, record_b = er.LIVE_PRESET  # edit a field to try your own

try:
    answer = judge.ask(*er.pair_request(record_a, record_b))
    p = answer.response.nouls["same"].noul
    print(
        f"p_same={p:.2f} → {bucket(p, LO, HI)}  ({answer.source}, {answer.latency_ms} ms)"
    )
except (NotRecorded, LiveUnavailable) as e:
    print(e)